In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [59]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

In [60]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [61]:
device

device(type='cuda')

In [62]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.hid_dim = hid_dim  # 은닉층 차원(context 벡터의 크기)
        self.n_layers = n_layers # Lstm 층 수

        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True) #(배치, 시퀀스, 특성)
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # (배치 크기, 원문 길이)
        embedded = self.dropout(self.embedding(src))
        # 2. LSTM을 통해 순방향 전파
        # outputs: 모든 시간대의 hidden state (Attention용)
        # hidden, cell: 마지막 시간대의 상태 (컨텍스트 벡터 역할)
        outputs, (hidden, cell) = self.rnn(embedded)

        return outputs, hidden, cell

In [63]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim  # 출력 어휘 크기
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, emb_dim)
        # LSTM: 이전 단어와 컨텍스트를 받아 다음 상태 계산
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
         # 출력층: LSTM 출력을 어휘 크기의 확률 분포로 변환
        self.fc_out = nn.Linear(hid_dim, output_dim)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, input, hidden, cell):
        # input: [batch_size] (현재 시간대의 단어, 1개씩 처리)
        
        # 1. 입력 차원 확장: [batch_size] -> [batch_size, 1]
        # LSTM은 (batch, seq_len, feature) 형태를 기대하므로 seq_len=1로 추가
        input = input.unsqueeze(1)
        # 2. 임베딩 적용
        embedded = self.dropout(self.embedding(input))
        # embedded: [batch_size, 1, emb_dim]
        
        # 3. LSTM 통과
        # hidden, cell: 인코더에서 전달받거나 이전 시간대의 상태
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output: [batch_size, 1, hid_dim]
        
        # 4. 선형층을 통해 어휘 크기의 벡터로 변환
        prediction = self.fc_out(output.squeeze(1))
        # prediction: [batch_size, output_dim] (각 단어의 확률)
        
        return prediction, hidden, cell


In [65]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: [batch_size, src_len] (원문)
        # trg: [batch_size, trg_len] (목표 문장, 번역문)
        # teacher_forcing_ratio: 정답 단어를 사용할 확률 (학습 안정화용)
        
        batch_size = trg.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim
        
        # 출력을 저장할 텐서 초기화
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        
        # 1. 인코더를 통해 컨텍스트 벡터(hidden, cell) 생성
        _, hidden, cell = self.encoder(src)
        
        # 2. 디코더의 첫 입력은 <START> 토큰 (보통 인덱스 0 또는 1)
        input = trg[:, 0]  # 모든 배치의 첫 번째 토큰
        
        # 3. 목표 문장 길이만큼 반복하며 단어 생성
        for t in range(1, trg_len):
            # 디코더로 다음 단어 예측
            output, hidden, cell = self.decoder(input, hidden, cell)
            
            # 예측 결과 저장
            outputs[:, t, :] = output
            
            # Teacher Forcing: 정답을 사용할지, 예측값을 사용할지 결정
            # 학습 초기에는 정답을 많이 사용하고, 나중에는 모델 예측을 더 사용
            teacher_force = random.random() < teacher_forcing_ratio
            
            # 가장 확률 높은 단어 선택 (greedy decoding)
            top1 = output.argmax(1)
            
            # 다음 입력 결정: 정답(trg[:, t]) 또는 예측값(top1)
            input = trg[:, t] if teacher_force else top1
        
        return outputs

In [66]:
import sentencepiece as spm
import json

In [9]:
with open("/kaggle/input/datasets/sinmunsu/kor2-jpn/ko2ja_kpop_1_training.json", "r") as f:
    data = json.load(f)


In [10]:
len(data)

160000

In [11]:
ko_corpus = "temp_kopus.txt"
jp_corpus = "temp_jppus.txt"
ko_src = open(ko_corpus, 'w')
jp_src = open(jp_corpus, 'w')
for x in data:
    ko_src.write(x['한국어'] + "\n")
    jp_src.write(x['일본어'] + "\n")
ko_src.close()
jp_src.close()

In [12]:
with open(jp_corpus, 'r') as f:
    for x in f:
        print(x)
        break

所属事務所側は、2人が親しいのは事実だが、結婚説は事実無根だと強調した。



In [ ]:
spm.SentencePieceTrainer.train(
    input=ko_corpus,
    model_prefix='bpe_model',
    vocab_size=32000,
    model_type='bpe',               # BPE 알고리즘 선택
    character_coverage=0.9995,
    max_sentencepiece_length=32,      # 더 긴 서브워드 허용
    split_by_whitespace=True,
    split_by_unicode_script=True,
    treat_whitespace_as_suffix=False, # 공백을 접두사로 (▁token 형태)
    allow_whitespace_only_pieces=True, # 공백만으로 된 토큰 허용
    normalization_rule_name='nmt_nfkc',
    remove_extra_whitespaces=True,
    input_sentence_size=10000000,     # 천만 문장만 사용
    shuffle_input_sentence=True,
    seed_sentencepiece_size=1000000,  # 초기 시드 크기
    shrinking_factor=0.75,            # EM 알고리즘 수축률 (unigram만)
    num_threads=4                    # 모든 코어 사용
)

In [1]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("ilhansevval/eng-fra")

print("Path to dataset files:", path)


Path to dataset files: /kaggle/input/datasets/ilhansevval/eng-fra


In [2]:
!ls -al /kaggle/input/datasets/ilhansevval/eng-fra

total 9324
drwxr-xr-x 2 nobody nogroup       0 Mar  4 05:14 .
drwxr-xr-x 3 root   root       4096 Mar  4 05:15 ..
-rw-r--r-- 1 nobody nogroup 9541158 Mar  4 05:14 eng-fra.txt


In [5]:
with open("/kaggle/input/datasets/ilhansevval/eng-fra/eng-fra.txt") as f:
    text = f.read().strip().split('\n')

In [7]:
text[:10]

['Go.\tVa !',
 'Run!\tCours\u202f!',
 'Run!\tCourez\u202f!',
 'Wow!\tÇa alors\u202f!',
 'Fire!\tAu feu !',
 "Help!\tÀ l'aide\u202f!",
 'Jump.\tSaute.',
 'Stop!\tÇa suffit\u202f!',
 'Stop!\tStop\u202f!',
 'Stop!\tArrête-toi !']

In [12]:
import re
def normalize_string(s):
    """간단한 텍스트 전처리: 소문자 변환 및 구두점 분리"""
    s = s.lower().strip()
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s.strip()

In [14]:
list(map(normalize_string,text[123].split('\t')))

['help us .', 'aide nous !']

In [15]:
class Vocabulary:
    def __init__(self):
        self.word2idx = {"<PAD>": PAD_IDX, "<SOS>": SOS_IDX, "<EOS>": EOS_IDX, "<UNK>": UNK_IDX}
        self.idx2word = {PAD_IDX: "<PAD>", SOS_IDX: "<SOS>", EOS_IDX: "<EOS>", UNK_IDX: "<UNK>"}
        self.idx = 4

    def add_sentence(self, sentence):
        for word in sentence.split(' '):
            if word not in self.word2idx:
                self.word2idx[word] = self.idx
                self.idx2word[self.idx] = word
                self.idx += 1

    def __len__(self):
        return len(self.word2idx)

In [16]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class TranslationDataset(Dataset):
    def __init__(self, file_path):
        self.src_sentences = []
        self.trg_sentences = []
        self.src_vocab = Vocabulary()
        self.trg_vocab = Vocabulary()
        
        # 파일 읽기 및 단어장 구축
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.read().strip().split('\n')
            
        for line in lines:
            parts = line.split('\t')
            if len(parts) >= 2:
                # 텍스트 전처리
                src = normalize_string(parts[0])
                trg = normalize_string(parts[1])
                
                self.src_sentences.append(src)
                self.trg_sentences.append(trg)
                
                self.src_vocab.add_sentence(src)
                self.trg_vocab.add_sentence(trg)

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, i):
        src_sentence = self.src_sentences[i]
        trg_sentence = self.trg_sentences[i]
        
        # 문자열을 인덱스 텐서로 변환 (<SOS>, <EOS> 추가)
        src_tensor = [SOS_IDX] + [self.src_vocab.word2idx.get(w, UNK_IDX) for w in src_sentence.split(' ')] + [EOS_IDX]
        trg_tensor = [SOS_IDX] + [self.trg_vocab.word2idx.get(w, UNK_IDX) for w in trg_sentence.split(' ')] + [EOS_IDX]
        
        return torch.tensor(src_tensor, dtype=torch.long), torch.tensor(trg_tensor, dtype=torch.long)

In [17]:
# 특수 토큰 정의
PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

In [18]:
dataset = TranslationDataset("/kaggle/input/datasets/ilhansevval/eng-fra/eng-fra.txt")

In [34]:
BATCH_SIZE = 128

def collate_fn(batch):
    src_batch, trg_batch = [], []
    
    for src_sample, trg_sample in batch:
        src_batch.append(src_sample)
        trg_batch.append(trg_sample)
        
    # batch_first=True 로 설정하여 [batch_size, seq_len] 형태로 패딩
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    trg_batch = pad_sequence(trg_batch, padding_value=PAD_IDX, batch_first=True)
    
    return src_batch, trg_batch

In [35]:
dataloader = DataLoader(
    dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn,
    drop_last=True
)

In [36]:
tmp = next(iter(dataloader))

In [43]:
tmp[0].shape

torch.Size([128, 17])

In [44]:
tmp[1].shape

torch.Size([128, 18])

In [51]:
tmp = next(iter(dataloader))

for i, (src_batch, trg_batch) in enumerate(dataloader):
    print(f"입력(src) 배치 크기: {src_batch.shape}")
    print(f"타겟(trg) 배치 크기: {trg_batch.shape}")
    break


입력(src) 배치 크기: torch.Size([128, 16])
타겟(trg) 배치 크기: torch.Size([128, 18])


In [67]:
INPUT_DIM = len(dataset.src_vocab) # 영어의 어휘 크기 
OUTPUT_DIM = len(dataset.trg_vocab) # 불어의 어휘 크기 
EMB_DIM = 256      # 임베딩 차원 (단어를 256차원 벡터로 표현)
HID_DIM = 512      # 은닉층 차원 (메모장의 용량)
N_LAYERS = 2       # LSTM 층 수 (깊이)
DROPOUT = 0.5      # 드롭아웃 비율

# 모델 인스턴스 생성
enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)

model = Seq2Seq(enc, dec, device).to(device)

# 손실 함수: 패딩 토큰 무시하고 계산 (ignore_index=0 가정)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# 옵티마이저: Adam 사용
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [68]:
def train(model, iterator, optimizer, criterion):
    model.train()  # 학습 모드 (드롭아웃 활성화)
    epoch_loss = 0
    
    for i, (src, trg) in enumerate(iterator):
        src, trg = src.to(device), trg.to(device)
        
        optimizer.zero_grad()  # 기울기 초기화
        
        # 순전파: teacher_forcing_ratio=0.5로 설정
        output = model(src, trg, 0.5)
        
        # output: [batch_size, trg_len, output_dim]
        # trg: [batch_size, trg_len]
        
        # 손실 계산을 위해 2D로 변환
        output_dim = output.shape[-1]
        output = output[:, 1:, :].reshape(-1, output_dim)  # <START> 제외
        trg = trg[:, 1:].reshape(-1)  # <START> 제외
        
        loss = criterion(output, trg)
        loss.backward()  # 역전파
        
        # 기울기 클리핑 (폭발 방지)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        
        optimizer.step()  # 가중치 업데이트
        
        epoch_loss += loss.item()
    
    return epoch_loss / len(iterator)

In [69]:
epochs = 10
for x in range(epochs):
    print(f"{x+1} epochs")
    train(model, dataloader, optimizer, criterion)

1 epochs
2 epochs
3 epochs
4 epochs
5 epochs
6 epochs
7 epochs
8 epochs
9 epochs
10 epochs


In [77]:
example_sentence = "i am running."
sentence = normalize_string(example_sentence)
tokens = sentence.split(' ')
src_vocab = dataset.src_vocab
token_indices  = [src_vocab.word2idx.get("<SOS>")]
for token in tokens:
    token_indices.append(src_vocab.word2idx.get(token, src_vocab.word2idx.get("<UNK>")))
token_indices.append(src_vocab.word2idx.get("<EOS>"))

In [78]:
token_indices

[1, 14, 93, 994, 5, 2]

In [79]:
src_tensor = torch.tensor(token_indices, dtype=torch.long).unsqueeze(0).to(device)

In [80]:
src_tensor

tensor([[  1,  14,  93, 994,   5,   2]], device='cuda:0')

In [81]:
model.eval()
with torch.no_grad():
    encoder_outputs, hidden, cell = model.encoder(src_tensor)

In [82]:
encoder_outputs

tensor([[[ 3.1855e-03,  5.2288e-02, -7.3828e-04,  ..., -2.7174e-02,
           4.6120e-03, -1.6557e-01],
         [-1.7524e-03,  4.5057e-03,  6.4005e-03,  ..., -7.0878e-03,
           1.0269e-02, -4.1563e-02],
         [-2.3540e-05,  6.1693e-04,  9.3309e-05,  ..., -5.3947e-02,
          -3.7278e-02, -7.6984e-02],
         [ 4.5191e-04,  4.6820e-03,  1.1721e-05,  ..., -1.1627e-01,
          -2.0353e-02, -2.8784e-02],
         [ 1.5408e-03,  7.5639e-03,  1.1198e-05,  ..., -2.6707e-02,
          -5.3862e-02, -7.5735e-02],
         [ 2.6444e-03,  2.3136e-02,  1.8129e-05,  ...,  2.4380e-02,
           8.9484e-03, -6.3287e-02]]], device='cuda:0')

In [85]:
trg_vocab = dataset.trg_vocab
trg_indices = [trg_vocab.word2idx.get("<SOS>")]
MAX_LEN = 50
for i in range(MAX_LEN):
    trg_tensor = torch.tensor([trg_indices[-1]], dtype=torch.long).to(device)
    with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
    pred_token = output.argmax(1).item()
    trg_indices.append(pred_token)
    if pred_token == trg_vocab.word2idx.get("<EOS>"):
        break

In [87]:
trg_indices

[1, 25, 27, 15, 2]

In [89]:
trg_tokens = []
for i in trg_indices:
    trg_tokens.append(trg_vocab.idx2word.get(i, "<UNK>"))

In [90]:
trg_tokens

['<SOS>', 'j', 'ai', '.', '<EOS>']

In [91]:
class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 2, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)
        
    def forward(self, hidden, encoder_outputs):
        # hidden: [batch_size, hid_dim] (디코더의 현재 상태)
        # encoder_outputs: [batch_size, src_len, hid_dim] (인코더의 모든 출력)
        
        src_len = encoder_outputs.shape[1]
        
        # hidden을 src_len만큼 반복: [batch_size, src_len, hid_dim]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        
        # 에너지(점수) 계산: [batch_size, src_len, hid_dim]
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        
        # Attention 가중치: [batch_size, src_len]
        attention = self.v(energy).squeeze(2)
        
        # 소프트맥스로 정규화 (모든 가중치의 합=1)
        return torch.softmax(attention, dim=1)


In [92]:
class AttnDecoder(nn.Module):
    """
    Attention을 사용하는 디코더
    """
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        
        self.embedding = nn.Embedding(output_dim, emb_dim)
        # 입력: 임베딩 + 컨텍스트 벡터(어텐션 결과)
        self.rnn = nn.LSTM(emb_dim + hid_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        # 출력: hid_dim -> output_dim (어휘 크기)
        self.fc_out = nn.Linear(emb_dim + hid_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell, encoder_outputs):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        # embedded: [batch_size, 1, emb_dim]
        
        # Attention 가중치 계산
        # hidden[-1]: LSTM의 마지막 층 hidden state 사용
        attn_weights = self.attention(hidden[-1], encoder_outputs)
        # attn_weights: [batch_size, src_len]
        
        # 가중치를 encoder_outputs에 적용하여 컨텍스트 벡터 생성
        attn_weights = attn_weights.unsqueeze(1)  # [batch_size, 1, src_len]
        context = torch.bmm(attn_weights, encoder_outputs)  # [batch_size, 1, hid_dim]
        
        # LSTM 입력: 임베딩 + 컨텍스트 연결
        rnn_input = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        
        # 출력 생성: 임베딩 + hidden + context를 결합하여 예측
        embedded = embedded.squeeze(1)
        output = output.squeeze(1)
        context = context.squeeze(1)
        
        prediction = self.fc_out(torch.cat((embedded, output, context), dim=1))
        
        return prediction, hidden, cell, attn_weights


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim
        
        # 텐서 생성 시 device 직접 지정
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size, device=self.device)
        
        # 인코더를 통과시켜 모든 타임스텝의 출력(encoder_outputs) 확보
        encoder_outputs, hidden, cell = self.encoder(src)
        
        # 디코더의 첫 입력은 <SOS> 토큰
        input = trg[:, 0]
        
        for t in range(1, trg_len):
            # 디코더에 encoder_outputs를 추가로 전달
            output, hidden, cell, _ = self.decoder(input, hidden, cell, encoder_outputs)
            
            outputs[:, t, :] = output
            
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            
            input = trg[:, t] if teacher_force else top1
            
        return outputs

In [93]:
attn = Attention(HID_DIM)
enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
dec = AttnDecoder(OUTPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT, attn)
model = Seq2Seq(enc, dec, device).to(device)


In [94]:
epochs = 10
loss = []
for x in range(epochs):
    print(f"{x+1} epochs")
    error = train(model, dataloader, optimizer, criterion)
    print(f"loss -> {error}")
    loss.append(error)


1 epochs
loss -> 9.909040664076018
2 epochs
loss -> 9.90941365140435
3 epochs
loss -> 9.908900348113237
4 epochs
loss -> 9.908890156112225
5 epochs
loss -> 9.909034179809742
6 epochs
loss -> 9.908821956264198
7 epochs
loss -> 9.90861734863051
8 epochs
loss -> 9.90928004262135
9 epochs
loss -> 9.90906902622435
10 epochs
loss -> 9.90947090691828
